# Fitting Delta Function with Burst Model

In [5]:
import os
import math
import numpy as np
import bagpipes as pipes
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib as mpl

Bagpipes: Latex turned off in rcParams, plots may look strange.
Bagpipes: PyMultiNest import failed, fitting will use the Nautilus sampler instead.


In [6]:
z = 4

import os
import math
import numpy as np
import bagpipes as pipes
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib as mpl

In [7]:
filter_folder = r"D:\Users\nina\Star_Formation_Histories\Filters"

filter_files = sorted([
    os.path.join(filter_folder, f)
    for f in os.listdir(filter_folder)
    if f.endswith(".txt")
])

global_min = np.inf
global_max = -np.inf

for ffile in filter_files:
    data = np.loadtxt(ffile)
    global_min = min(global_min, data[:,0].min())
    global_max = max(global_max, data[:,0].max())

print(f"Wavelength coverage: {global_min:.0f}–{global_max:.0f} Å")

Wavelength coverage: 3526–50996 Å


In [8]:
ages_myr_1 = np.arange(0,110,10)
ages_myr_2 = np.arange(200,1100,100)
years = np.concatenate([ages_myr_1, ages_myr_2]) / 1000

dust = {
    "type": "Calzetti",
    "Av": 0.2,
    "eta": 2.
}

nebular = {
    "logU": -3.
}

In [9]:
def burst_model(t, z, dust, nebular):
    return {
        "redshift": z,
        "burst": {"age": t, "massformed": 10, "metallicity": 0.5},
        "dust": dust,
        "nebular": nebular,
        "t_bc": 0.01,
        "veldisp": 200.
    }


def compute_filter_wavelengths(filter_files):
    eff_waves = []
    for ffile in filter_files:
        filt_data = np.loadtxt(ffile)
        wave = filt_data[:, 0]
        transmission = filt_data[:, 1]
        eff_wave = np.sum(wave * transmission) / np.sum(transmission)
        eff_waves.append(eff_wave)
    return np.array(eff_waves)


def build_models(years, z, dust, nebular, filter_files, wavelength_range):

    models = []

    for t in years:
        components = burst_model(t, z, dust, nebular)

        model = pipes.model_galaxy(
            components,
            filt_list=filter_files,
            spec_wavs=wavelength_range
        )

        models.append(model)

    return models

In [ ]:
wavelength_range = np.arange(
    global_min - 500,
    global_max + 500,
    5.
)

filter_wavelengths = compute_filter_wavelengths(filter_files)

models = build_models(
    years,
    z,
    dust,
    nebular,
    filter_files,
    wavelength_range
)

In [ ]:
SNR = 10

synthetic_flux = np.array([model.photometry for model in models])  # F_lambda

c = 2.99792458e18  # Å/s

flux_nu = synthetic_flux * (filter_wavelengths**2) / c
flux_nu = flux_nu / 1e-23      # Jy
flux_in_microjansky = flux_nu * 1e6

err_in_microjansky = flux_in_microjansky / SNR

In [ ]:
galaxy_values = []

for age in years:

    age_idx = np.argmin(np.abs(years - age))

    print(f"Fitting model with age = {years[age_idx]:.3f} Gyr")

    galaxy_photometry = np.c_[
        flux_in_microjansky[age_idx],
        err_in_microjansky[age_idx]
    ]

    galaxy = pipes.galaxy(
        str(years[age_idx]),
        lambda ID: galaxy_photometry,
        spectrum_exists=False,
        filt_list=filter_files
    )

    galaxy_values.append(galaxy)

## Fitting

In [ ]:
# Exponential SFH
exp = {
    "age": (0.001, 1.1),            # in Gyr
    "tau": (0.001, 5.),            # in Gyr
    "massformed": (5., 12.),      # log10(M*/M_sun)
    "metallicity": (0.0001, 2),      # Z/Z_sun
    "metallicity_prior": "log_10"
}

# Dust
dust_fit = {
    "type": "Calzetti",
    "Av": (0., 2.)
}

# Fit instructions
fit_instructions = {
    "redshift": 2.0,
    "exponential": exp,
    "dust": dust_fit,
    "nebular": {"logU": (-4., -2.)}
}

In [ ]:
fits = []

for i, galaxy in enumerate(galaxy_values):

    print(f"Running fit {i+1}/{len(galaxy_values)}  (Age = {years[i]:.3f} Gyr)")

    fit = pipes.fit(galaxy, fit_instructions)
    fit.fit(verbose=True)

    fits.append(fit)

print("All fits finished.")

In [ ]:
for i, fit in enumerate(fits):

    print(f"\nPlotting results for age {years[i]:.3f} Gyr")

    fit.plot_spectrum_posterior(show=True, save=False)

    fit.plot_sfh_posterior(show=True, save=False)

    fit.plot_corner(show=True, save=False)